# LFA Detection — Lateral Flow Assay Strip Quantification

End-to-end pipeline: **strip photo → YOLO marker detection → intensity profile → control + 6 band intensities → per-band concentration regressors → patient condition classifier.**

```
Strip photo ─▶ YOLO (C, C, 1 markers) ─▶ strip geometry ─▶ intensity profile
          ─▶ control + 6 band intensities ─▶ 6 regressors (concentration)
          ─▶ condition classifier (5-fold CV)
```

### Contents
1. [Setup & configuration](#1.-Setup-&-configuration)
2. [Core helpers: detection, signal extraction, plotting](#2.-Core-helpers)
3. [Step 1 — Calibrate band positions](#3.-Step-1-—-Calibrate-band-positions)
4. [Step 2 — Build the training dataset](#4.-Step-2-—-Build-the-training-dataset)
5. [Step 3 — Train the per-band regressors](#5.-Step-3-—-Train-the-per-band-regressors)
6. [Step 4 — Predict patient samples (incl. positive/negative control images)](#6.-Step-4-—-Predict-patient-samples)
7. [Step 5 — Explore the patient predictions](#7.-Step-5-—-Explore-the-patient-predictions)
8. [Step 6 — Classify patient condition](#8.-Step-6-—-Classify-patient-condition)
9. [Step 7 — Review misclassified images](#9.-Step-7-—-Review-misclassified-images)

### How to use this notebook
- Each step **saves its output to disk** and the next step **reads it back**, so you can restart the kernel and resume from any section.
- Function-definition cells are collapsed (`#@title` in Colab). Expand them to read the code; you normally just run them.
- **Edit paths and parameters only in section 1.**

> ℹ️ **One extraction method everywhere.** Training data (section 4) and patient images (section 6) use the *same* `sample_signal_using_distance`: each band's intensity is the **maximum signal inside its ±3σ window** (see section 2.2).

## 1. Setup & configuration

In [ ]:
# Install dependencies. Needed on Colab; comment these out if your environment already has them.
%pip install -q ultralytics opencv-python seaborn scikit-learn
%pip install -q umap-learn   # optional, only for the UMAP plot in section 8

In [ ]:
import json
import logging
import math
import os
import sys
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks, savgol_filter
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, mean_absolute_error,
                             mean_squared_error, r2_score)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
log = logging.getLogger("lfa")

### 1.1 Paths

`BASE_DIR` is `/content` on Colab and the current folder elsewhere. Expected layout:

```
BASE_DIR/
├── best.pt                  # YOLO weights (detects markers "C" and "1")
├── calibration/{1..6}/      # calibration images, one folder per concentration
├── patient_samples/         # patient strip images (listed in patient_readings.csv)
├── positive_control/        # ALL images here are analysed, labelled "positive"
├── negative_control/        # ALL images here are analysed, labelled "negative"
├── patient_readings.csv     # columns: Image, Condition, (optional) Notes -- lists the patient_samples/ images
└── outputs/                 # everything this notebook generates
```

In [ ]:
IN_COLAB = "google.colab" in sys.modules
BASE_DIR = Path("/content") if IN_COLAB else Path(".").resolve()

OUT_DIR = BASE_DIR / "outputs"

P = {
    "yolo_weights":       BASE_DIR / "best.pt",
    "calibration_dir":    BASE_DIR / "calibration",
    "patient_images_dir": BASE_DIR / "patient_samples",
    "patient_csv":        BASE_DIR / "patient_readings.csv",
    # Folders searched (in order) for the images listed in patient_readings.csv
    "image_dirs": [BASE_DIR / "patient_samples",
                   BASE_DIR / "positive_control",
                   BASE_DIR / "negative_control"],
    # Every image in these folders is analysed and labelled with the given Condition
    # (they do NOT need to be listed in patient_readings.csv).
    "labeled_image_dirs": {BASE_DIR / "positive_control": "positive",
                           BASE_DIR / "negative_control": "Negative"},
    "calibration_json":   OUT_DIR / "calibration.json",
    "datasets_dir":       OUT_DIR / "datasets",
    "models_dir":         OUT_DIR / "models",
    "results_dir":        OUT_DIR / "results",
}
for key in ("datasets_dir", "models_dir", "results_dir"):
    P[key].mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)

### 1.2 Parameters

Everything tunable lives here. Defaults match the original notebook.

In [ ]:
cfg = {
    "geometry": {
        "real_world_distance_cm": 2.55,   # known distance between the "1" marker and the nearest control marker
        "num_bands": 6,
    },
    "signal": {
        "track_width": 15,                # pixels averaged perpendicular to the sampling path
        "tolerance": 30,                  # pixel padding before/after the path
        "savgol_window": 21,
        "savgol_polyorder": 2,
        "baseline_sigma": 20,             # gaussian baseline removal
        "peak_width": 4,                  # min peak width (px) for control / first-pass peaks
        "first_pass_prominence": 0.02,    # fraction of max signal (calibration pass)
        "control_prominence": 0.05,       # fraction of max signal (control peak)
    },
    "concentrations": [1, 2, 3, 4, 5, 6],  # sub-folders of calibration/
    "regressor": {                         # per-band concentration regressors
        "n_splits": 5, "epochs": 150, "batch_size": 32,
        "lr": 0.001, "huber_delta": 1.0, "seed": 42,
    },
    "classifier": {                        # patient condition classifier
        "n_splits": 5, "epochs": 200, "lr": 0.001,
        "weight_decay": 0.0001, "dropout": 0.2, "seed": 42,
        # Labels differing only by case (Negative / negative) are merged automatically.
        # Use this to merge *different* words, e.g. {"positive": "HIV1"}. Empty = no extra merging.
        "condition_aliases": {},
    },
}

### 1.3 Unpack data (optional)

If your data is still zipped in `BASE_DIR`, this extracts it. Folders that already exist are skipped.

In [ ]:
for name in ["calibration.zip", "patient_samples.zip", "positive_control.zip", "negative_control.zip"]:
    zip_path = BASE_DIR / name
    target = BASE_DIR / zip_path.stem
    if target.exists():
        print(f"{target.name}/ already present - skipping")
    elif zip_path.is_file():
        zipfile.ZipFile(zip_path).extractall(BASE_DIR)
        print(f"Extracted {name}")
    else:
        print(f"{name} not found - skipping")

## 2. Core helpers

### 2.1 Marker detection & strip geometry

YOLO must find exactly **two control markers (`C`)** and **one signal marker (`1`)**. Their positions give the sampling line along the strip (midpoint, slope, pixel distance); the known real-world spacing converts pixels to cm.

In [ ]:
#@title Detection & geometry helpers
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


class DetectionError(Exception):
    """Raised when the expected markers are not found in an image."""


def load_yolo(weights_path):
    """Load YOLO weights. Returns ``(model, class_names)``."""
    from ultralytics import YOLO  # imported lazily so the rest of the package works without it

    model = YOLO(str(weights_path))
    return model, model.names


def return_box_location(cls, boxes, class_names):
    """Return the ``xywh`` boxes whose predicted class name equals ``cls``."""
    class_location = []
    for box in boxes:
        class_id = int(box.cls[0].item())
        if class_names[class_id] == cls:
            class_location.append(box.xywh[0].tolist())
    return class_location


def check_prediction_classes(prediction, class_names):
    """Require exactly two control (``C``) markers and one ``1`` marker."""
    boxes = prediction.boxes
    boxes_c = return_box_location("C", boxes, class_names)
    boxes_1 = return_box_location("1", boxes, class_names)
    if len(boxes_c) != 2:
        raise DetectionError("Incorrect Number of Control Signal ID")
    if len(boxes_1) != 1:
        raise DetectionError("Incorrect Number of 1 Signal ID")
    return boxes_c, boxes_1


def roi_slope_center(boxes_c, boxes_1):
    """Derive the sampling geometry from marker positions.

    Returns
    -------
    midpoint : (x, y) midpoint between the two control markers
    slope    : slope of the line from the nearest control marker to the "1" marker
    distance : pixel distance between that control marker and the "1" marker
    target   : (x, y) of the "1" marker
    """
    c1 = boxes_c[0]
    c2 = boxes_c[1]
    midpt_c = (c1[0] + c2[0]) // 2, (c1[1] + c2[1]) // 2

    dist_c1_1 = np.sqrt((c1[0] - boxes_1[0][0]) ** 2 + (c1[1] - boxes_1[0][1]) ** 2)
    slope_c1_1 = (c1[1] - boxes_1[0][1]) / (c1[0] - boxes_1[0][0])
    dist_c2_1 = np.sqrt((c2[0] - boxes_1[0][0]) ** 2 + (c2[1] - boxes_1[0][1]) ** 2)
    slope_c2_1 = (c2[1] - boxes_1[0][1]) / (c2[0] - boxes_1[0][0])

    target = (boxes_1[0][0], boxes_1[0][1])
    if dist_c1_1 < dist_c2_1:
        return midpt_c, slope_c1_1, dist_c1_1, target
    return midpt_c, slope_c2_1, dist_c2_1, target


def locate_strip(prediction, class_names):
    """Convenience wrapper: YOLO result -> ``(midpoint, slope, distance, target)``."""
    boxes_c, boxes_1 = check_prediction_classes(prediction, class_names)
    return roi_slope_center(boxes_c, boxes_1)


def calculate_real_world_distance(px_distance, peak, rw_distance=2.55):
    """Convert a pixel offset ``peak`` to centimetres.

    ``px_distance`` is the pixel length that corresponds to ``rw_distance`` cm.
    """
    cm_per_pixel = rw_distance / px_distance
    return cm_per_pixel * peak

### 2.2 Intensity profile & band extraction

The green channel is averaged across the strip width along the sampling line, inverted (dark bands → peaks), smoothed (Savitzky–Golay) and baseline-corrected (Gaussian).

- `sample_signal` — **first pass:** finds every peak (used for calibration; should yield 7 = control + 6 bands).
- `sample_signal_using_distance` — **second pass:** finds the control peak, then searches for each band within ±3σ of its calibrated position.

> **Same method for training and patients.** Training data (section 4) and patient images (section 6) go through this one function, so the regressors always see intensities computed identically.
>
> **Scaling note:** the strip is averaged over `arange(-(w//2), w//2)` pixel columns but divided by `w`. For the default odd `track_width=15` that is 14 columns / 15, so intensities are scaled by 14/15. That is consistent everywhere, but if you ever change it you must retrain (sections 4 → 8).

In [ ]:
#@title Signal extraction
def _sampling_direction(midpoint, target, slope):
    """Unit vector along the strip, oriented from ``midpoint`` toward ``target``."""
    angle = math.atan(slope)
    vx, vy = math.cos(angle), math.sin(angle)

    dx = target[0] - midpoint[0]
    dy = target[1] - midpoint[1]
    if vx * dx + vy * dy < 0:
        angle += math.pi
        vx, vy = math.cos(angle), math.sin(angle)
    return angle, vx, vy


def _track_offsets(track_width):
    """Perpendicular pixel offsets for averaging across the strip width."""
    offsets = np.arange(-(track_width // 2), track_width // 2)
    return offsets, track_width


def sample_track(img, midpoint, target, slope, distance, tolerance, track_width,
                 min_steps=0):
    """Sample the (green channel) image along the strip axis.

    Returns ``(avg_signal, x_path, y_path)`` where ``avg_signal`` is the raw
    intensity averaged across the track width at each step along the path.
    """
    gray = img[:, :, 1]
    h, w = gray.shape

    angle, vx, vy = _sampling_direction(midpoint, target, slope)

    x_start = midpoint[0] - tolerance * vx
    y_start = midpoint[1] - tolerance * vy
    x_end = midpoint[0] + (distance + tolerance) * vx
    y_end = midpoint[1] + (distance + tolerance) * vy

    num_steps = int(abs(distance) + 2 * tolerance)
    if min_steps:
        num_steps = max(min_steps, num_steps)
    x_path = np.linspace(x_start, x_end, num_steps)
    y_path = np.linspace(y_start, y_end, num_steps)

    nx, ny = -math.sin(angle), math.cos(angle)
    offsets, divisor = _track_offsets(track_width)

    signal_sum = np.zeros(num_steps, dtype=float)
    for offset in offsets:
        sample_x = np.clip((x_path + offset * nx).astype(int), 0, w - 1)
        sample_y = np.clip((y_path + offset * ny).astype(int), 0, h - 1)
        signal_sum += gray[sample_y, sample_x]

    return signal_sum / divisor, x_path, y_path


def clean_signal(avg_signal, savgol_window=21, savgol_polyorder=2, baseline_sigma=20):
    """Invert, smooth and baseline-correct a raw intensity profile.

    Dark bands become positive peaks; slow background drift is removed with a
    wide gaussian baseline.
    """
    inverted = np.max(avg_signal) - avg_signal
    win = min(savgol_window, len(inverted) // 2 * 2 - 1)
    smooth = savgol_filter(inverted, win, savgol_polyorder) if win >= 3 else inverted
    baseline = gaussian_filter(smooth, sigma=baseline_sigma)
    return np.maximum(smooth - baseline, 0)


def sample_signal(img, midpoint, target, slope, distance, tolerance=5, track_width=30,
                  prominence_frac=0.02, peak_width=4, savgol_window=21,
                  savgol_polyorder=2, baseline_sigma=20, show=False):
    """Detect all peaks along the strip.

    Returns ``(peak_indices, peak_intensities)``. For a valid calibration image
    there should be exactly 7 peaks: the control plus six bands.
    """
    avg_signal, x_path, y_path = sample_track(
        img, midpoint, target, slope, distance, tolerance, track_width,
    )
    final_signal = clean_signal(avg_signal, savgol_window, savgol_polyorder, baseline_sigma)

    peaks, _ = find_peaks(
        final_signal, prominence=np.max(final_signal) * prominence_frac, width=peak_width
    )

    if show:
        plot_first_pass(img, x_path, y_path, midpoint, slope, final_signal, peaks)

    return peaks, final_signal[peaks]


def sample_signal_using_distance(img, midpoint, target, slope, distance,
                                 real_world_peak_locations, real_world_peak_std,
                                 tolerance=30, track_width=15, rw_distance=2.55,
                                 control_prominence=0.05, peak_width=4,
                                 savgol_window=21, savgol_polyorder=2,
                                 baseline_sigma=20, show=False):
    """Quantify the control and six band intensities using known band distances.

    Each band's intensity is the maximum processed signal inside a +/- 3 sigma
    window around its calibrated position (relative to the control peak).

    Parameters
    ----------
    real_world_peak_locations, real_world_peak_std
        Mean / std of each band's position (cm) relative to the control peak,
        from ``lfa.calibration``.

    Returns
    -------
    peak_locations : int array (7,)   pixel index of control + 6 bands
    peak_intensities : float array (7,)   processed intensity of control + 6 bands
    """
    if distance <= 0:
        raise ValueError("Calibration distance must be positive.")

    avg_signal, x_path, y_path = sample_track(
        img, midpoint, target, slope, distance, tolerance, track_width, min_steps=1,
    )
    final_signal = clean_signal(avg_signal, savgol_window, savgol_polyorder, baseline_sigma)

    cm_per_pixel = rw_distance / distance
    n = len(final_signal)

    # Control peak = first prominent peak from the left.
    peaks, _ = find_peaks(
        final_signal,
        prominence=max(float(np.max(final_signal)) * control_prominence, 1e-8),
        width=peak_width,
    )
    if len(peaks) == 0:
        raise ValueError("No control peak detected.")

    control_idx = int(peaks[0])
    peak_locations = [control_idx]
    peak_intensities = [float(final_signal[control_idx])]
    windows = []

    for location_cm, std_cm in zip(real_world_peak_locations, real_world_peak_std):
        expected_idx = control_idx + location_cm / cm_per_pixel
        std_px = std_cm / cm_per_pixel

        window_start = max(0, int(np.floor(expected_idx - 3 * std_px)))
        window_end = min(n - 1, int(np.ceil(expected_idx + 3 * std_px)))
        windows.append((expected_idx - 3 * std_px, expected_idx + 3 * std_px))

        if window_start <= window_end:
            # Band intensity = maximum signal inside the window.
            window = final_signal[window_start:window_end + 1]
            peak_idx = window_start + int(np.argmax(window))
            peak_intensity = float(final_signal[peak_idx])
        else:  # window fell entirely outside the signal
            peak_idx = int(np.clip(round(expected_idx), 0, n - 1))
            peak_intensity = 0.0

        peak_locations.append(peak_idx)
        peak_intensities.append(peak_intensity)

    peak_locations = np.asarray(peak_locations, dtype=int)
    peak_intensities = np.asarray(peak_intensities, dtype=float)

    if show:
        plot_band_extraction(
            img, x_path, y_path, midpoint, slope, final_signal,
            peak_locations, peak_intensities, windows,
        )

    return peak_locations, peak_intensities

### 2.3 Plotting helpers

In [ ]:
#@title Plotting helpers
def _finish(fig, save_path=None, show=False, dpi=150):
    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)


def plot_first_pass(img, x_path, y_path, midpoint, slope, final_signal, peaks,
                    save_path=None, show=True):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax1.plot(x_path, y_path, color="lime", lw=2, label="Path (with tolerance)")
    ax1.scatter(*midpoint, color="magenta", s=100, label="Midpoint C (Ref)", zorder=5)
    ax1.set_title(f"Sampling Path following Slope {slope:.3f}")
    ax1.legend()

    ax2.plot(final_signal, color="blue", lw=2, label="Cleaned Signal")
    ax2.fill_between(range(len(final_signal)), final_signal, alpha=0.2, color="blue")
    if len(peaks) > 0:
        ax2.plot(peaks, final_signal[peaks], "x", color="red", markersize=12, mew=2,
                 label="Detected Peaks")
        for p in peaks:
            ax2.text(p, final_signal[p], f"{final_signal[p]:.1f}", color="red", weight="bold")
    ax2.set_title("Quantified Intensity with Peak Detection")
    ax2.set_ylabel("Intensity (AU)")
    ax2.set_xlabel("Pixels along path (including start/end buffer)")
    ax2.legend()

    _finish(fig, save_path, show)


def plot_band_extraction(img, x_path, y_path, midpoint, slope, final_signal,
                         peak_locations, peak_intensities, windows,
                         save_path=None, show=True):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax1.plot(x_path, y_path, color="lime", lw=2, label="Sampling Path")
    ax1.scatter(*midpoint, color="magenta", s=100, label="Reference Midpoint", zorder=5)
    ax1.set_title(f"Sampling Path (Slope {slope:.3f})")
    ax1.legend()

    ax2.plot(final_signal, color="blue", lw=2, label="Processed Signal")
    ax2.plot(peak_locations[0], peak_intensities[0], "rx", markersize=12, mew=2,
             label="Control Peak")
    ax2.annotate(f"Control: {peak_intensities[0]:.1f}",
                 (peak_locations[0], peak_intensities[0]))

    last = len(final_signal) - 1
    for i, (lo, hi) in enumerate(windows, start=1):
        ax2.axvspan(max(0, lo), min(last, hi), alpha=0.15)
        ax2.plot(peak_locations[i], peak_intensities[i], "x", markersize=10)
        ax2.annotate(f"Band {i}: {peak_intensities[i]:.1f}",
                     (peak_locations[i], peak_intensities[i]), fontsize=9)

    ax2.set_title("Control-Referenced Band Intensities")
    ax2.set_ylabel("Intensity (AU)")
    ax2.set_xlabel("Pixels along sampling path")
    ax2.legend()

    _finish(fig, save_path, show)


def plot_patient_diagnostic(img, name, peak_intensities, save_path):
    """Per-patient diagnostic: image next to a bar chart of control + band intensities."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(str(name))
    axes[0].axis("off")

    axes[1].bar(["Control"] + [f"Band {i}" for i in range(1, 7)], peak_intensities[:7])
    axes[1].set_ylabel("Processed intensity (AU)")
    axes[1].tick_params(axis="x", rotation=45)

    _finish(fig, save_path, show=False)


def plot_regression_diagnostics(target, ratio, true, pred, out_dir=None, prefix="", show=False):
    """Three plots: concentration vs ratio with NN fit, predicted vs actual, residuals."""
    out_dir = Path(out_dir) if out_dir else None
    sep = f"{prefix}_" if prefix else ""
    idx = np.argsort(pred)

    fig = plt.figure(figsize=(10, 6))
    plt.scatter(target, ratio, alpha=0.6, label="Observed")
    plt.plot(pred[idx], np.asarray(ratio)[idx], linewidth=3, label="NN Fit (Final Model)")
    plt.xlabel("Concentration")
    plt.ylabel("Signal / Control")
    plt.title(f"Concentration vs Ratio {prefix}".strip())
    plt.legend()
    plt.grid()
    _finish(fig, out_dir / f"{sep}concentration_vs_ratio.png" if out_dir else None, show)

    fig = plt.figure(figsize=(6, 6))
    plt.scatter(true, pred, alpha=0.7)
    lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
    plt.plot(lims, lims, "--")
    plt.xlabel("True Concentration")
    plt.ylabel("Predicted Concentration")
    plt.title(f"Predicted vs Actual {prefix}".strip())
    plt.grid()
    _finish(fig, out_dir / f"{sep}predicted_vs_actual.png" if out_dir else None, show)

    fig = plt.figure(figsize=(8, 5))
    plt.scatter(true, pred - true, alpha=0.7)
    plt.axhline(0, linestyle="--")
    plt.xlabel("True Concentration")
    plt.ylabel("Residual")
    plt.title(f"Residual Plot {prefix}".strip())
    plt.grid()
    _finish(fig, out_dir / f"{sep}residuals.png" if out_dir else None, show)


def plot_correlation_heatmap(successful, prediction_cols, save_path=None, show=True):
    import seaborn as sns

    fig = plt.figure(figsize=(8, 6))
    sns.heatmap(successful[prediction_cols].corr(), annot=True, cmap="coolwarm",
                vmin=-1, vmax=1, fmt=".2f")
    plt.title("Correlation Between Band Predictions")
    _finish(fig, save_path, show)


def plot_violin_by_condition(successful, prediction_cols, save_path=None, show=True):
    """Violin plot of the six band predictions for each patient condition."""
    import seaborn as sns

    plot_data = successful.melt(
        id_vars=["Condition"], value_vars=prediction_cols,
        var_name="Band", value_name="Prediction",
    )
    plot_data["Band"] = plot_data["Band"].str.replace("_Prediction", "", regex=False)

    fig = plt.figure(figsize=(14, 7))
    sns.violinplot(data=plot_data, x="Condition", y="Prediction", hue="Band",
                   inner="quartile", cut=0, density_norm="width")
    plt.title("Distribution of Band Predictions by Patient Condition")
    plt.xlabel("Patient Condition")
    plt.ylabel("Predicted Value")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Band", bbox_to_anchor=(1.02, 1), loc="upper left")
    _finish(fig, save_path, show)


def plot_confusion_matrix(cm, classes, save_path=None, show=False):
    import seaborn as sns

    fig = plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted Condition")
    plt.ylabel("Actual Condition")
    plt.title("5-Fold CV: Out-of-Fold Confusion Matrix")
    _finish(fig, save_path, show, dpi=300)


def plot_umap_embeddings(embeddings, labels, classes, save_path=None, show=False,
                         random_state=42):
    """UMAP of the held-out 16-d classifier embeddings (requires ``umap-learn``)."""
    import umap  # optional dependency

    coords = umap.UMAP(n_components=2, random_state=random_state).fit_transform(embeddings)
    fig = plt.figure(figsize=(8, 6))
    for i, name in enumerate(classes):
        m = labels == i
        plt.scatter(coords[m, 0], coords[m, 1], label=name, alpha=0.8)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title("UMAP of Held-Out Classifier Embeddings (5-Fold CV)")
    plt.legend()
    _finish(fig, save_path, show)


def plot_misclassified(misclassified, image_dirs, save_path=None, show=False, ncols=3):
    """Grid of every image misclassified in out-of-fold predictions.

    ``image_dirs`` is a folder or an ordered list of folders to search.
    """
    if misclassified.empty:
        print("No misclassified images in the out-of-fold predictions.")
        return

    image_dirs = [Path(image_dirs)] if isinstance(image_dirs, (str, Path)) else [Path(d) for d in image_dirs]
    n_images = len(misclassified)
    nrows = math.ceil(n_images / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False)

    for ax, (_, row) in zip(axes.flat, misclassified.iterrows()):
        filename = str(row["Image"]).strip()
        exact = row.get("Image_Path", "")
        if isinstance(exact, str) and exact and Path(exact).is_file():
            path = Path(exact)  # exact file (keeps same-named images in different folders apart)
        else:
            path = find_image(filename, image_dirs)
        image = cv2.imread(str(path)) if path is not None else None

        if image is None:
            msg = "Could not read image" if path is not None else f"Image not found\n{filename}"
            ax.text(0.5, 0.5, msg, ha="center", va="center")
            ax.axis("off")
            continue

        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{filename}\nActual: {row['Actual_Class']}\nPredicted: {row['Predicted_Class']}",
                     fontsize=10, color="red")
        ax.axis("off")

    for ax in axes.flat[n_images:]:
        ax.axis("off")

    plt.suptitle("Misclassified Images — 5-Fold Cross-Validation")
    _finish(fig, save_path, show)

### 2.4 Patient CSV & image lookup

The images analysed in section 6 come from two sources:

1. **Rows in `patient_readings.csv`** — found by filename (searched in `patient_samples/` first).
2. **Every image in the labelled control folders** — `positive_control/` (label `positive`) and `negative_control/` (label `negative`). These need not be in the CSV and keep their exact file path, so same-named files in different folders never get mixed up.

These helpers build the combined table and report which folder each image came from.

In [ ]:
#@title Patient CSV & image lookup
def as_dirs(image_dirs):
    """Accept one folder or an ordered iterable of folders; return a list of Paths."""
    if isinstance(image_dirs, (str, Path)):
        return [Path(image_dirs)]
    return [Path(d) for d in image_dirs]


def list_images(directory):
    """Sorted image paths in ``directory`` (skips hidden files / non-images)."""
    directory = Path(directory)
    return [
        str(directory / f)
        for f in sorted(os.listdir(directory))
        if not f.startswith(".") and Path(f).suffix.lower() in IMAGE_EXTENSIONS
    ]


def unify_case(labels):
    """Treat labels that differ only by case (or surrounding spaces) as the same label.

    The first spelling encountered is used for every variant, e.g. ``Negative`` and
    ``negative`` both become whichever appeared first. Other text is unchanged, so
    ``HIV1`` and ``positive`` stay different. Missing values and ``""`` are preserved.
    """
    series = pd.Series(labels)
    canonical = {}
    out = []
    for v in series:
        if isinstance(v, str):
            v = v.strip()
            key = v.casefold()
            if key:
                canonical.setdefault(key, v)
                v = canonical[key]
        out.append(v)
    return pd.Series(out, index=series.index)


def load_patient_csv(patient_csv):
    """Read ``patient_readings.csv`` and normalise its columns.

    Requires ``Image``. Adds an empty ``Condition`` if absent. An unlabeled
    third column (pandas names it ``Unnamed: 2``) is treated as ``Notes``.
    """
    patients = pd.read_csv(patient_csv)

    if "Image" not in patients.columns:
        raise ValueError(f"Expected 'Image' column. Found {patients.columns.tolist()}")
    if "Condition" not in patients.columns:
        patients["Condition"] = ""
    if "Unnamed: 2" in patients.columns:
        patients["Notes"] = patients["Unnamed: 2"].fillna("")
    elif "Notes" not in patients.columns:
        patients["Notes"] = ""

    patients["Image"] = patients["Image"].astype(str).str.strip()
    return patients


def find_all(image_name, image_dirs):
    """Every file matching ``image_name``: direct hits first (folder order), then subfolders."""
    image_name = str(image_name).strip()
    if not image_name or image_name.lower() == "nan":
        return []

    dirs = as_dirs(image_dirs)
    candidate = Path(image_name)
    if candidate.is_absolute():
        return [candidate] if candidate.is_file() else []

    found = []
    for d in dirs:
        direct = d / candidate
        if direct.is_file():
            found.append(direct)
    for d in dirs:
        if d.is_dir():
            for p in d.rglob(candidate.name):
                if p.is_file() and p not in found:
                    found.append(p)
    return found


def find_image(image_name, image_dirs):
    """First match for ``image_name`` (see ``find_all``), or ``None``."""
    matches = find_all(image_name, image_dirs)
    if len(matches) > 1:
        log.warning("Multiple files named %s; using %s", Path(str(image_name)).name, matches[0])
    return matches[0] if matches else None


def build_patient_table(patient_csv, image_dirs, labeled_dirs=None):
    """CSV rows plus every image found in the labeled folders.

    Parameters
    ----------
    patient_csv : path to ``patient_readings.csv``
    image_dirs : folders searched (in order) for the filenames listed in the CSV
    labeled_dirs : ``{folder: label}`` -- every image in ``folder`` is added as a row
        with ``Condition = label``. If the CSV already lists that exact file, the CSV
        row is kept and the folder copy is skipped (no double counting).

    Condition labels that differ only by case are unified (see ``unify_case``).

    Returns a DataFrame with ``Image``, ``Condition``, ``Notes``, ``Path`` (exact file
    for folder rows, empty for CSV rows) and ``Source`` (``csv`` or the folder name).
    """
    patients = load_patient_csv(patient_csv)
    patients["Path"] = ""
    patients["Source"] = "csv"
    listed = set(patients["Image"])

    rows = []
    for folder, label in (labeled_dirs or {}).items():
        folder = Path(folder)
        if not folder.is_dir():
            log.warning("Labeled image folder not found: %s", folder)
            continue
        for path in list_images(folder):
            name = Path(path).name
            if name in listed:
                hit = find_image(name, image_dirs)
                if hit is not None and Path(hit).resolve() == Path(path).resolve():
                    log.info("%s is already listed in the CSV; skipping folder copy", name)
                    continue
            rows.append({"Image": name, "Condition": label, "Notes": "",
                         "Path": path, "Source": folder.name})

    if rows:
        patients = pd.concat([patients, pd.DataFrame(rows)], ignore_index=True)
    # CSV rows come first, so the CSV's spelling (e.g. "Negative") wins over the folder
    # label ("negative"); the two are then one class everywhere downstream.
    patients["Condition"] = unify_case(patients["Condition"])
    return patients


def check_image_lookup(patient_csv, image_dirs, labeled_dirs=None):
    """Report where every image to be analysed is found.

    Returns a DataFrame with ``Image``, ``Condition``, ``Notes``, ``Source``,
    ``Found_In`` (folder of the first match, or ``MISSING``), ``N_Matches``,
    ``Duplicate_Name`` (same filename used by more than one row) and ``All_Matches``.
    """
    table = build_patient_table(patient_csv, image_dirs, labeled_dirs)
    dup_names = table["Image"].duplicated(keep=False)

    rows = []
    for i, row in table.iterrows():
        matches = [Path(row["Path"])] if row["Path"] else find_all(row["Image"], image_dirs)
        rows.append({
            "Image": row["Image"],
            "Condition": row["Condition"],
            "Notes": row["Notes"],
            "Source": row["Source"],
            "Found_In": matches[0].parent.name if matches else "MISSING",
            "N_Matches": len(matches),
            "Duplicate_Name": bool(dup_names.iloc[i]),
            "All_Matches": "; ".join(str(m) for m in matches),
        })
    return pd.DataFrame(rows)


def summarize_lookup(report):
    """Print a short summary of ``check_image_lookup`` output."""
    print(f"Total images to analyse: {len(report)}")
    print("\nRows by source (csv = listed in patient_readings.csv):")
    print(report["Source"].value_counts().to_string())
    print("\nCondition x folder:")
    print(pd.crosstab(report["Condition"], report["Found_In"]).to_string())

    missing = report[report["Found_In"] == "MISSING"]
    dupes = report[report["N_Matches"] > 1]
    same_name = report[report["Duplicate_Name"]]
    print(f"\nMissing: {len(missing)} | CSV filename found in more than one folder: {len(dupes)}"
          f" | Same filename used by several rows: {len(same_name)}")
    if len(missing):
        print("  missing ->", ", ".join(missing["Image"].tolist()))
    if len(dupes):
        print("  in several folders ->", ", ".join(dupes["Image"].tolist()))
    if len(same_name):
        print("  same filename in several rows (kept apart by exact path) ->",
              ", ".join(sorted(set(same_name["Image"]))))
    return missing, dupes

## 3. Step 1 — Calibrate band positions

For every calibration image, find the 7 peaks and measure each band's distance from the control peak in **cm**. The mean ± std per band is saved to `calibration.json` and defines where to look for each band later.

Images that fail marker detection or don't show exactly 7 peaks are skipped (and logged).

In [ ]:
#@title Calibration helpers
NUM_BANDS = 6


def measure_band_positions(yolo_model, class_names, calibration_dir, concentrations, cfg):
    """Measure each band's distance from the control peak on every calibration image.

    Returns ``{band: [position_cm, ...]}`` for bands 1..6. Images that fail
    detection or do not show exactly 7 peaks are skipped (and logged).
    """
    sig = cfg["signal"]
    rw = cfg["geometry"]["real_world_distance_cm"]
    positions = {b: [] for b in range(1, NUM_BANDS + 1)}

    for concentration in concentrations:
        base_dir = Path(calibration_dir) / str(concentration)
        log.info("Measuring band positions in %s", base_dir)
        files = list_images(base_dir)
        results = yolo_model(files)

        for path, result in zip(files, results):
            try:
                midpoint, slope, distance, target = locate_strip(result, class_names)
                img = cv2.imread(path)
                peak_locations, _ = sample_signal(
                    img, midpoint, target, slope, distance,
                    track_width=sig["track_width"], tolerance=sig["tolerance"],
                    prominence_frac=sig["first_pass_prominence"],
                    peak_width=sig["peak_width"],
                    savgol_window=sig["savgol_window"],
                    savgol_polyorder=sig["savgol_polyorder"],
                    baseline_sigma=sig["baseline_sigma"],
                )
                if len(peak_locations) != 7:
                    raise DetectionError(f"Expected 7 peaks, found {len(peak_locations)}")

                control = peak_locations[0]
                for band in range(1, NUM_BANDS + 1):
                    px = peak_locations[band] - control
                    positions[band].append(calculate_real_world_distance(distance, px, rw))
            except (DetectionError, ZeroDivisionError) as err:
                log.warning("Skipped %s: %s", path, err)

    return positions


def summarize_band_positions(positions):
    """Mean / (population) std of each band's position in cm."""
    return {
        "mean": [float(np.mean(positions[b])) for b in range(1, NUM_BANDS + 1)],
        "std": [float(np.std(positions[b])) for b in range(1, NUM_BANDS + 1)],
        "n_images": len(positions[1]),
    }


def save_calibration(path, summary, rw_distance_cm):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "band_position_cm_mean": summary["mean"],
        "band_position_cm_std": summary["std"],
        "n_images": summary["n_images"],
        "real_world_distance_cm": rw_distance_cm,
    }
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)


def load_calibration(path):
    """Return ``(band_means_cm, band_stds_cm)`` as lists of six floats."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Calibration file not found: {path}. Run scripts/01_calibrate_bands.py first."
        )
    with open(path) as f:
        payload = json.load(f)
    return payload["band_position_cm_mean"], payload["band_position_cm_std"]

**Sanity check on one calibration image** (sampling path + all detected peaks). Skip if you like.

In [ ]:
yolo, class_names = load_yolo(P["yolo_weights"])

sig = cfg["signal"]
example = list_images(P["calibration_dir"] / str(cfg["concentrations"][0]))[0]
img = cv2.imread(example)
midpoint, slope, distance, target = locate_strip(yolo(example)[0], class_names)

peaks, intensities = sample_signal(
    img, midpoint, target, slope, distance,
    track_width=sig["track_width"], tolerance=sig["tolerance"],
    prominence_frac=sig["first_pass_prominence"], peak_width=sig["peak_width"],
    show=True,
)
print(f"{example}: {len(peaks)} peaks detected (expected 7)")

In [ ]:
positions = measure_band_positions(
    yolo, class_names, P["calibration_dir"], cfg["concentrations"], cfg
)
summary = summarize_band_positions(positions)
if summary["n_images"] == 0:
    raise RuntimeError("No calibration image produced 7 peaks - check data and YOLO weights.")

save_calibration(P["calibration_json"], summary, cfg["geometry"]["real_world_distance_cm"])

display(pd.DataFrame({
    "Band": range(1, 7),
    "Mean position from control (cm)": summary["mean"],
    "Std (cm)": summary["std"],
}).round(4))
print(f"Used {summary['n_images']} calibration images -> {P['calibration_json']}")

## 4. Step 2 — Build the training dataset

Using the calibrated band positions, extract the **(control, signal)** intensity pair of each band for every calibration image. This produces one table per band with columns `Control`, `Signal`, `Target` (the concentration).

Uses the same extraction as patient images (section 2.2).

In [ ]:
#@title Training-pair extraction
def extract_training_pairs(yolo_model, class_names, calibration_dir, concentrations,
                           band_means, band_stds, cfg):
    """Extract (control, signal) intensities for each band at each concentration.

    Returns a dict ``{band: DataFrame[Control, Signal, Target]}`` where
    ``Target`` is the concentration.
    """
    sig = cfg["signal"]
    rw = cfg["geometry"]["real_world_distance_cm"]
    rows = {b: [] for b in range(1, NUM_BANDS + 1)}

    for concentration in concentrations:
        base_dir = Path(calibration_dir) / str(concentration)
        log.info("Extracting intensities in %s", base_dir)
        files = list_images(base_dir)
        results = yolo_model(files)

        for path, result in zip(files, results):
            try:
                midpoint, slope, distance, target = locate_strip(result, class_names)
                img = cv2.imread(path)
                _, intensities = sample_signal_using_distance(
                    img, midpoint, target, slope, distance,
                    list(band_means), list(band_stds),
                    track_width=sig["track_width"], tolerance=sig["tolerance"],
                    rw_distance=rw,
                    control_prominence=sig["control_prominence"],
                    peak_width=sig["peak_width"],
                    savgol_window=sig["savgol_window"],
                    savgol_polyorder=sig["savgol_polyorder"],
                    baseline_sigma=sig["baseline_sigma"],
                )
                for band in range(1, NUM_BANDS + 1):
                    rows[band].append((intensities[0], intensities[band], concentration))
            except (DetectionError, ValueError, ZeroDivisionError) as err:
                log.warning("Skipped %s: %s", path, err)

    return {
        band: pd.DataFrame(r, columns=["Control", "Signal", "Target"])
        for band, r in rows.items()
    }

In [ ]:
band_means, band_stds = load_calibration(P["calibration_json"])

datasets = extract_training_pairs(
    yolo, class_names, P["calibration_dir"], cfg["concentrations"],
    band_means, band_stds, cfg,
)

for band, df in datasets.items():
    df.to_csv(P["datasets_dir"] / f"band{band}.csv", index=False)

display(pd.DataFrame({f"Band {b}": df.groupby("Target").size() for b, df in datasets.items()})
        .rename_axis("Concentration"))
display(datasets[1].head())

## 5. Step 3 — Train the per-band regressors

One small MLP (`2 → 64 → 32 → 16 → 1`) per band maps raw `[Control, Signal]` intensities to concentration.

For each band: **5-fold CV** for an honest error estimate (MAE / RMSE / R²), then a **final model refit on all data**, saved with its input/output scalers as `final_model_band{N}.pth`.

In [ ]:
#@title Regressor model, training & checkpoints
class Net(nn.Module):
    """2 -> 64 -> 32 -> 16 -> 1 MLP."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)


def _fit(model, loader, epochs, lr, huber_delta):
    criterion = nn.HuberLoss(delta=huber_delta)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()


def train_regressor_cv(df, target_col="Target", control_col="Control", signal_col="Signal",
                       n_splits=5, epochs=150, batch_size=32, lr=0.001, huber_delta=1.0,
                       seed=42, exclude=None, plot_dir=None, plot_prefix="", show_plots=False,
                       verbose=True):
    """K-fold CV, then refit on all data.

    Returns a dict with ``best_fold``, ``fold_results`` (array of MAE/RMSE/R2),
    ``best_model``, ``final_model``, ``scaler_X``, ``scaler_y``, ``pred``, ``true``.
    """
    df_clean = df[~df[target_col].isin(exclude or [])].copy()

    X = df_clean[[control_col, signal_col]].values
    y = df_clean[target_col].values.reshape(-1, 1)

    if seed is not None:
        torch.manual_seed(seed)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_results, fold_models, fold_scalers = [], [], []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        if verbose:
            print(f"\n===== FOLD {fold + 1} =====")

        scaler_X, scaler_y = StandardScaler(), StandardScaler()
        X_train = torch.FloatTensor(scaler_X.fit_transform(X[train_idx]))
        y_train = torch.FloatTensor(scaler_y.fit_transform(y[train_idx]))
        X_val = torch.FloatTensor(scaler_X.transform(X[val_idx]))

        loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
        model = Net()
        _fit(model, loader, epochs, lr, huber_delta)

        model.eval()
        with torch.no_grad():
            pred_scaled = model(X_val).numpy()

        pred = scaler_y.inverse_transform(pred_scaled).flatten()
        true = y[val_idx].flatten()

        mae = mean_absolute_error(true, pred)
        rmse = np.sqrt(mean_squared_error(true, pred))
        r2 = r2_score(true, pred)
        if verbose:
            print(f"MAE : {mae:.4f}\nRMSE: {rmse:.4f}\nR²  : {r2:.4f}")

        fold_results.append((mae, rmse, r2))
        fold_models.append(model)
        fold_scalers.append((scaler_X, scaler_y))

    best_idx = int(np.argmin([r[0] for r in fold_results]))
    best_model = fold_models[best_idx]
    if verbose:
        print(f"\nBest fold: {best_idx + 1}")

    # Final model: refit on all data
    scaler_X_full, scaler_y_full = StandardScaler(), StandardScaler()
    X_full_t = torch.FloatTensor(scaler_X_full.fit_transform(X))
    y_full_t = torch.FloatTensor(scaler_y_full.fit_transform(y))
    full_loader = DataLoader(TensorDataset(X_full_t, y_full_t), batch_size=batch_size, shuffle=True)

    final_model = Net()
    _fit(final_model, full_loader, epochs, lr, huber_delta)

    final_model.eval()
    with torch.no_grad():
        pred_scaled = final_model(X_full_t).numpy()
    pred = scaler_y_full.inverse_transform(pred_scaled).flatten()
    true = y.flatten()

    if plot_dir is not None or show_plots:
        ratio = df_clean[signal_col] / df_clean[control_col]
        plot_regression_diagnostics(df_clean[target_col], ratio, true, pred,
                                    out_dir=plot_dir, prefix=plot_prefix, show=show_plots)

    fold_results = np.array(fold_results)
    if verbose:
        print("\n==============================")
        print("        FINAL CV RESULTS       ")
        print("==============================")
        for i, name in enumerate(["MAE ", "RMSE", "R²  "]):
            print(f"{name}: {fold_results[:, i].mean():.4f} ± {fold_results[:, i].std():.4f}")
        print("==============================\n")

    return {
        "best_fold": best_idx + 1,
        "fold_results": fold_results,
        "best_model": best_model,
        "final_model": final_model,
        "scaler_X": scaler_X_full,
        "scaler_y": scaler_y_full,
        "pred": pred,
        "true": true,
    }


def band_model_path(models_dir, band):
    return Path(models_dir) / f"final_model_band{band}.pth"


def save_checkpoint(path, results):
    """Save the final model + scaler statistics from ``train_regressor_cv``."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_state_dict": results["final_model"].state_dict(),
        "scaler_X_mean": results["scaler_X"].mean_.tolist(),
        "scaler_X_scale": results["scaler_X"].scale_.tolist(),
        "scaler_y_mean": results["scaler_y"].mean_.tolist(),
        "scaler_y_scale": results["scaler_y"].scale_.tolist(),
    }, str(path))


def load_band_model(model_path, device="cpu"):
    """Load a saved band regressor and its scalers (validated)."""
    model_path = Path(model_path)
    if not model_path.is_file():
        raise FileNotFoundError(f"Model not found: {model_path}")

    checkpoint = torch.load(str(model_path), map_location=device, weights_only=True)

    net = Net().to(device)
    net.load_state_dict(checkpoint["model_state_dict"])
    net.eval()

    for key in ("scaler_X_mean", "scaler_X_scale", "scaler_y_mean", "scaler_y_scale"):
        if key not in checkpoint:
            raise KeyError(f"{model_path.name} missing {key}")

    X_mean = np.asarray(checkpoint["scaler_X_mean"], dtype=np.float32).reshape(1, -1)
    X_scale = np.asarray(checkpoint["scaler_X_scale"], dtype=np.float32).reshape(1, -1)
    y_mean = np.asarray(checkpoint["scaler_y_mean"], dtype=np.float32).reshape(-1)
    y_scale = np.asarray(checkpoint["scaler_y_scale"], dtype=np.float32).reshape(-1)

    if X_mean.shape[1] != 2:
        raise ValueError(f"{model_path.name}: expected 2 input features, got {X_mean.shape[1]}")
    if np.any(X_scale == 0):
        raise ValueError(f"Invalid input scaler in {model_path.name}")
    if np.any(y_scale == 0):
        raise ValueError(f"Invalid output scaler in {model_path.name}")

    return {"model": net, "X_mean": X_mean, "X_scale": X_scale,
            "y_mean": y_mean, "y_scale": y_scale}


def predict_band(control_intensity, signal_intensity, model_info):
    """Predict a concentration from raw ``[Control, Signal]`` (training units/order)."""
    X = np.array([[control_intensity, signal_intensity]], dtype=np.float32)
    X_scaled = (X - model_info["X_mean"]) / model_info["X_scale"]

    with torch.no_grad():
        y_scaled = model_info["model"](torch.tensor(X_scaled, dtype=torch.float32)).cpu().numpy()

    prediction = y_scaled * model_info["y_scale"] + model_info["y_mean"]
    return float(prediction.reshape(-1)[0])

In [ ]:
reg = cfg["regressor"]
plot_dir = P["results_dir"] / "regressor_plots"
metrics = []

for band in range(1, cfg["geometry"]["num_bands"] + 1):
    print(f"\n{'=' * 30}\n Training band {band}\n{'=' * 30}")
    df = pd.read_csv(P["datasets_dir"] / f"band{band}.csv")

    results = train_regressor_cv(
        df, n_splits=reg["n_splits"], epochs=reg["epochs"], batch_size=reg["batch_size"],
        lr=reg["lr"], huber_delta=reg["huber_delta"], seed=reg["seed"],
        plot_dir=plot_dir, plot_prefix=f"band{band}", show_plots=True,
    )
    save_checkpoint(band_model_path(P["models_dir"], band), results)

    fr = results["fold_results"]
    metrics.append({"Band": band, "Best fold": results["best_fold"],
                    "MAE": fr[:, 0].mean(), "MAE std": fr[:, 0].std(),
                    "RMSE": fr[:, 1].mean(), "RMSE std": fr[:, 1].std(),
                    "R2": fr[:, 2].mean(), "R2 std": fr[:, 2].std()})

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(P["results_dir"] / "regressor_cv_metrics.csv", index=False)
display(metrics_df.round(4))

## 6. Step 4 — Predict patient samples

`PatientPipeline` bundles YOLO, the calibrated band positions and the six regressors. For every image — the rows of `patient_readings.csv` **plus every image in `positive_control/` and `negative_control/`** (source folder recorded in `Image_Dir`) — it returns control/band intensities, band/control ratios and **predicted concentrations**. A failure on one image never stops the batch; it is recorded in the `Status` / `Error` columns.

Uses the same extraction as the training data (section 2.2).

### 6.1 Check where each CSV image is found

Run this **before** predicting. It builds the full list of images (CSV rows + every image in the control folders) and reports where each was found, any missing files, and filenames used more than once. Check that the total is what you expect (e.g. 61 CSV rows + the images in the two control folders).

In [ ]:
lookup = check_image_lookup(P["patient_csv"], P["image_dirs"], P["labeled_image_dirs"])
missing, dupes = summarize_lookup(lookup)
lookup.to_csv(P["results_dir"] / "data_check.csv", index=False)

if len(missing):
    display(missing[["Image", "Condition"]])

### 6.2 Load the pipeline

In [ ]:
#@title PatientPipeline
class PatientPipeline:
    def __init__(self, yolo_weights, calibration_json, models_dir, cfg,
                 plot_dir=None, device="cpu"):
        self.cfg = cfg
        self.plot_dir = Path(plot_dir) if plot_dir else None
        if self.plot_dir:
            self.plot_dir.mkdir(parents=True, exist_ok=True)

        if not Path(yolo_weights).is_file():
            raise FileNotFoundError(f"YOLO model not found: {yolo_weights}")
        self.yolo, self.class_names = load_yolo(yolo_weights)
        self.band_means, self.band_stds = load_calibration(calibration_json)

        self.band_models = {}
        for band in range(1, NUM_BANDS + 1):
            path = band_model_path(models_dir, band)
            self.band_models[band] = load_band_model(path, device)
            log.info("Loaded band %d: %s", band, path.name)

    # -- image lookup (delegates to lfa.data_check) ------------------------
    find_image = staticmethod(find_image)

    # -- single image -------------------------------------------------------
    def process_image(self, image_path, image_name, condition="", notes=""):
        result = {"Image": str(image_name), "Condition": condition, "Notes": notes,
                  "Image_Dir": Path(image_path).parent.name,  # which folder it was found in
                  "Image_Path": str(image_path),
                  "Status": "ERROR", "Error": ""}
        sig = self.cfg["signal"]

        try:
            img = cv2.imread(str(image_path))
            if img is None:
                raise ValueError(f"Could not read image: {image_path}")

            detections = self.yolo(str(image_path))
            if len(detections) == 0:
                raise ValueError("YOLO returned no detections.")

            midpoint, slope, distance, target = locate_strip(detections[0], self.class_names)

            _, intensities = sample_signal_using_distance(
                img, midpoint, target, slope, distance,
                self.band_means, self.band_stds,
                track_width=sig["track_width"], tolerance=sig["tolerance"],
                rw_distance=self.cfg["geometry"]["real_world_distance_cm"],
                control_prominence=sig["control_prominence"],
                peak_width=sig["peak_width"],
                savgol_window=sig["savgol_window"],
                savgol_polyorder=sig["savgol_polyorder"],
                baseline_sigma=sig["baseline_sigma"],
            )
            if len(intensities) < 7:
                raise ValueError(f"Expected 7 peak intensities; got {len(intensities)}")

            control = float(intensities[0])
            result["Control_Intensity"] = control

            for band in range(1, NUM_BANDS + 1):
                signal = float(intensities[band])
                result[f"Band_{band}_Intensity"] = signal
                result[f"Band_{band}_Ratio"] = signal / control if control != 0 else np.nan

                # Models are trained on raw [Control, Signal], not on the ratio alone.
                if not np.isfinite(control) or not np.isfinite(signal):
                    raise ValueError(f"Non-finite intensity for band {band}")
                result[f"Band_{band}_Prediction"] = predict_band(
                    control, signal, self.band_models[band]
                )

            if self.plot_dir:
                plot_path = self.plot_dir / (
                    f"{Path(image_path).parent.name}_{Path(image_name).stem}_diagnostic.png"
                )
                plot_patient_diagnostic(img, image_name, intensities, plot_path)
                result["Diagnostic_Plot"] = str(plot_path)

            result["Status"] = "OK"

        except Exception as error:  # noqa: BLE001 - one bad image must not stop the batch
            result["Error"] = f"{type(error).__name__}: {error}"

        return result

    # -- whole CSV ----------------------------------------------------------
    def run(self, patient_csv, image_dirs, labeled_dirs=None):
        """Process every CSV image plus every image in ``labeled_dirs``.

        ``image_dirs``  -- folder(s) searched for the filenames in the CSV.
        ``labeled_dirs`` -- ``{folder: label}``; each image in each folder becomes a row
        labelled ``label`` (e.g. ``{positive_control: "positive"}``).
        """
        for d in as_dirs(image_dirs) + [Path(p) for p in (labeled_dirs or {})]:
            if not d.is_dir():
                log.warning("Image folder not found: %s", d)
        patients = build_patient_table(patient_csv, image_dirs, labeled_dirs)

        all_results = []
        for index, patient in patients.iterrows():
            image_name = patient["Image"]
            condition = patient.get("Condition", "")
            notes = patient.get("Notes", "")

            log.info("[%d/%d] Processing %s", index + 1, len(patients), image_name)
            if patient["Path"]:  # folder row: exact file known
                image_path = Path(patient["Path"])
            else:
                image_path = self.find_image(image_name, image_dirs)

            if image_path is None:
                result = {"Image": image_name, "Condition": condition, "Notes": notes,
                          "Image_Dir": "", "Image_Path": "", "Status": "ERROR",
                          "Error": f"Image not found: {image_name}"}
            else:
                result = self.process_image(image_path, image_name, condition, notes)

            if result["Status"] == "ERROR":
                log.warning("  %s", result["Error"])
            all_results.append(result)

        return pd.DataFrame(all_results)

### 6.3 Optional — inspect a single image

Sampling path, ±3σ search windows and the maximum found in each.

In [ ]:
band_means, band_stds = load_calibration(P["calibration_json"])

IMAGE_TO_INSPECT = list_images(P["patient_images_dir"])[0]   # <- or set a path yourself, e.g. "/content/patient_samples/IMG_0001.jpg"
img = cv2.imread(IMAGE_TO_INSPECT)
midpoint, slope, distance, target = locate_strip(yolo(IMAGE_TO_INSPECT)[0], class_names)

_, intensities = sample_signal_using_distance(
    img, midpoint, target, slope, distance, band_means, band_stds,
    track_width=sig["track_width"], tolerance=sig["tolerance"],
    rw_distance=cfg["geometry"]["real_world_distance_cm"],
    control_prominence=sig["control_prominence"], peak_width=sig["peak_width"],
    show=True,
)

print(f"Image: {IMAGE_TO_INSPECT}")
print(f"Control intensity: {intensities[0]:.3f}")
for i in range(1, 7):
    print(f"Band {i}: intensity {intensities[i]:.3f} | ratio to control {intensities[i] / intensities[0]:.4f}")

### 6.4 Predict all images

In [ ]:
patient_dir = P["results_dir"] / "patient_results"
patient_dir.mkdir(parents=True, exist_ok=True)

pipeline = PatientPipeline(
    yolo_weights=P["yolo_weights"],
    calibration_json=P["calibration_json"],
    models_dir=P["models_dir"],
    cfg=cfg,
    plot_dir=patient_dir / "diagnostic_plots",
)
patient_results = pipeline.run(P["patient_csv"], P["image_dirs"],
                                labeled_dirs=P["labeled_image_dirs"])

PREDICTIONS_CSV = patient_dir / "patient_predictions.csv"
patient_results.to_csv(PREDICTIONS_CSV, index=False)

print("Results saved to:", PREDICTIONS_CSV)
display(patient_results["Status"].value_counts(dropna=False).to_frame("Images"))
display(patient_results.head())

## 7. Step 5 — Explore the patient predictions

Quality checks and distributions of the six predicted concentrations.

In [ ]:
PREDICTIONS_CSV = P["results_dir"] / "patient_results" / "patient_predictions.csv"
df = pd.read_csv(PREDICTIONS_CSV)
prediction_cols = [f"Band_{i}_Prediction" for i in range(1, 7)]

print("Dataset shape:", df.shape)
print("\nProcessing status:")
print(df["Status"].value_counts(dropna=False))
print("\nCondition counts:")
print(df["Condition"].value_counts(dropna=False))
print("\nCondition x source folder:")
print(pd.crosstab(df["Condition"], df["Image_Dir"].replace("", "(not found)")))

successful = df[df["Status"] == "OK"].copy()
failed = df[df["Status"] != "OK"].copy()
print(f"\nTotal images: {len(df)} | Successful: {len(successful)} | Failed: {len(failed)}")

if not failed.empty:
    display(failed[["Image", "Condition", "Error"]])

In [ ]:
print("Missing predictions:")
display(successful[prediction_cols].isna().sum().to_frame("Missing"))
display(successful[prediction_cols].describe().T)

**Correlation between the six band predictions**

In [ ]:
plot_correlation_heatmap(successful, prediction_cols, show=True)

**Distribution of each band's prediction per patient condition**

In [ ]:
plot_violin_by_condition(successful, prediction_cols, show=True)

## 8. Step 6 — Classify patient condition

An MLP classifies patient condition from the six predicted concentrations. Evaluation uses **stratified 5-fold CV**: scaling is fit on each training fold only, and every image gets an **out-of-fold** prediction, so the reported performance is on data the model never trained on.

Requires at least 5 labelled samples in every condition.

> **Labels.** Labels that differ only by case (`Negative` / `negative`) are merged into one class automatically. Different words, such as `HIV1` (CSV) and `positive` (control folder), stay separate classes. The class table printed below shows exactly which classes the classifier sees. To merge different words, set `condition_aliases` in section 1.2, e.g. `{"positive": "HIV1"}`.

In [ ]:
#@title Classifier & cross-validation
FEATURE_COLS = [f"Band_{i}_Prediction" for i in range(1, 7)]


class ConditionClassifier(nn.Module):
    def __init__(self, input_size=6, num_classes=2, dropout=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, num_classes),
        )

    def forward(self, x):
        return self.network(x)


def load_classification_data(csv_path, feature_cols=FEATURE_COLS, condition_aliases=None):
    """Read ``patient_predictions.csv`` and keep clean, labelled rows.

    Labels that differ only by case (``Negative`` / ``negative``) are always treated as
    one class. ``condition_aliases`` optionally merges *different* labels, e.g.
    ``{"positive": "HIV1"}``. Returns ``(data, X, y, encoder)``.
    """
    df = pd.read_csv(csv_path)
    data = df.loc[df["Status"].eq("OK")].copy()
    data["Condition"] = data["Condition"].astype("string").str.strip()
    if condition_aliases:
        data["Condition"] = data["Condition"].replace(condition_aliases)
    data["Condition"] = unify_case(data["Condition"]).astype("string")
    data[feature_cols] = data[feature_cols].apply(pd.to_numeric, errors="coerce")
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna(subset=feature_cols + ["Condition"])
    data = data[data["Condition"] != ""].reset_index(drop=True)

    X = data[feature_cols].to_numpy(dtype=np.float32)
    encoder = LabelEncoder()
    y = encoder.fit_transform(data["Condition"].astype(str))
    return data, X, y, encoder


def cross_validate_classifier(X, y, classes, n_splits=5, epochs=200, lr=0.001,
                              weight_decay=1e-4, dropout=0.2, seed=42, device=None,
                              verbose=True):
    """Stratified K-fold CV (preprocessing fit on the training fold only).

    Returns a dict with ``oof_predictions``, ``oof_probabilities``,
    ``oof_embeddings``, ``fold_metrics`` (DataFrame) and ``report`` (str).
    """
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_classes = len(classes)
    class_counts = np.bincount(y)
    if class_counts.min() < n_splits:
        raise ValueError(
            f"{n_splits}-fold stratified CV requires at least {n_splits} samples in every "
            f"class. Counts: {dict(zip(classes, class_counts))}"
        )

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_predictions = np.full(len(y), -1, dtype=int)
    oof_probabilities = np.zeros((len(y), n_classes))
    oof_embeddings = np.full((len(y), 16), np.nan)
    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        if verbose:
            print(f"\n{'=' * 20} FOLD {fold}/{n_splits} {'=' * 20}")

        y_train, y_test = y[train_idx], y[test_idx]
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[train_idx]).astype(np.float32)
        X_test = scaler.transform(X[test_idx]).astype(np.float32)

        X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
        X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
        y_train_t = torch.tensor(y_train, dtype=torch.long, device=device)

        torch.manual_seed(seed + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed + fold)

        model = ConditionClassifier(X.shape[1], n_classes, dropout).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        for _ in range(epochs):
            model.train()
            optimizer.zero_grad()
            loss = criterion(model(X_train_t), y_train_t)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            probabilities = torch.softmax(model(X_test_t), dim=1)
            predictions = probabilities.argmax(dim=1).cpu().numpy()
            embeddings = model.network[:-1](X_test_t).cpu().numpy()  # penultimate layer (16-d)

        oof_predictions[test_idx] = predictions
        oof_probabilities[test_idx] = probabilities.cpu().numpy()
        oof_embeddings[test_idx] = embeddings

        fold_results.append({
            "Fold": fold,
            "Accuracy": accuracy_score(y_test, predictions),
            "Balanced_Accuracy": balanced_accuracy_score(y_test, predictions),
            "Macro_F1": f1_score(y_test, predictions, average="macro", zero_division=0),
        })
        if verbose:
            r = fold_results[-1]
            print(f"Accuracy:          {r['Accuracy']:.4f}")
            print(f"Balanced accuracy: {r['Balanced_Accuracy']:.4f}")
            print(f"Macro F1:          {r['Macro_F1']:.4f}")

    fold_df = pd.DataFrame(fold_results)
    report = classification_report(
        y, oof_predictions, labels=np.arange(n_classes), target_names=classes, zero_division=0
    )
    if verbose:
        print("\nPer-fold metrics:")
        print(fold_df.round(4).to_string(index=False))
        print("\nMean and standard deviation across folds:")
        print(fold_df.drop(columns="Fold").agg(["mean", "std"]).round(4))
        print("\nOverall out-of-fold classification report:")
        print(report)

    return {
        "oof_predictions": oof_predictions,
        "oof_probabilities": oof_probabilities,
        "oof_embeddings": oof_embeddings,
        "fold_metrics": fold_df,
        "report": report,
        "confusion_matrix": confusion_matrix(y, oof_predictions, labels=np.arange(n_classes)),
    }


def _id_cols(data):
    return ["Image", "Condition"] + [c for c in ("Image_Dir", "Image_Path") if c in data.columns]


def build_prediction_table(data, y, oof_predictions, oof_probabilities, encoder):
    """Per-image out-of-fold predictions with class probabilities."""
    table = data[_id_cols(data)].copy().reset_index(drop=True)
    table["Actual_Class"] = encoder.inverse_transform(y)
    table["Predicted_Class"] = encoder.inverse_transform(oof_predictions)
    table["Correct"] = table["Actual_Class"] == table["Predicted_Class"]
    for i, name in enumerate(encoder.classes_):
        table[f"Probability_{name}"] = oof_probabilities[:, i]
    return table


def find_misclassified(data, y, oof_predictions, encoder, feature_cols=FEATURE_COLS):
    """Rows the out-of-fold classifier got wrong, with their six predicted concentrations."""
    error_df = data[_id_cols(data)].copy().reset_index(drop=True)
    error_df["Actual_Class"] = encoder.inverse_transform(y)
    error_df["Predicted_Class"] = encoder.inverse_transform(oof_predictions)
    error_df["Correct"] = error_df["Actual_Class"] == error_df["Predicted_Class"]
    for col in feature_cols:
        error_df[col] = data[col].to_numpy()

    misclassified = (
        error_df.loc[~error_df["Correct"]]
        .sort_values(["Actual_Class", "Predicted_Class", "Image"])
        .reset_index(drop=True)
    )
    return error_df, misclassified

In [ ]:
clf = cfg["classifier"]
cv_dir = P["results_dir"] / "patient_results" / "cv5"
cv_dir.mkdir(parents=True, exist_ok=True)

data, X, y, encoder = load_classification_data(
    PREDICTIONS_CSV, condition_aliases=clf["condition_aliases"]
)
classes = encoder.classes_
print("Samples:", len(X), "| Classes:", dict(zip(classes, np.bincount(y))))

cv = cross_validate_classifier(
    X, y, classes, n_splits=clf["n_splits"], epochs=clf["epochs"], lr=clf["lr"],
    weight_decay=clf["weight_decay"], dropout=clf["dropout"], seed=clf["seed"],
)

cv["fold_metrics"].to_csv(cv_dir / "fold_metrics.csv", index=False)
build_prediction_table(data, y, cv["oof_predictions"], cv["oof_probabilities"], encoder).to_csv(
    cv_dir / "oof_predictions.csv", index=False)
np.save(cv_dir / "oof_embeddings.npy", cv["oof_embeddings"])

In [ ]:
plot_confusion_matrix(cv["confusion_matrix"], classes,
                      save_path=cv_dir / "oof_confusion_matrix.png", show=True)

**Optional — UMAP of the held-out classifier embeddings** (needs `umap-learn`).

In [ ]:
plot_umap_embeddings(cv["oof_embeddings"], y, classes,
                     save_path=cv_dir / "umap_embeddings.png", show=True,
                     random_state=clf["seed"])

## 9. Step 7 — Review misclassified images

Every image the out-of-fold classifier got wrong, with its six predicted concentrations, plus a gallery of the strips themselves. Useful for spotting bad photos, mislabelled samples or detection failures.

In [ ]:
error_df, misclassified = find_misclassified(data, y, cv["oof_predictions"], encoder)

print(f"Total images: {len(error_df)}")
print(f"Correct: {error_df['Correct'].sum()}")
print(f"Incorrect: {len(misclassified)}")
print(f"Error rate: {(~error_df['Correct']).mean():.2%}")

display(misclassified)
misclassified.to_csv(cv_dir / "misclassified_images.csv", index=False)

In [ ]:
plot_misclassified(misclassified, P["image_dirs"],
                   save_path=cv_dir / "misclassified_images.png", show=True)

In [ ]:
import math
import matplotlib.image as mpimg

PER_PAGE = 4      # examples per figure (all pages are shown automatically)

# --- attach each misclassified image's saved diagnostic plot (from step 4) ---
preds = pd.read_csv(PREDICTIONS_CSV)
key = "Image_Path" if ("Image_Path" in misclassified.columns and "Image_Path" in preds.columns) else "Image"
diag_lookup = preds.drop_duplicates(key).set_index(key)["Diagnostic_Plot"]

mis = misclassified.copy()
mis["Diagnostic_Plot"] = mis[key].map(diag_lookup)

n_pages = max(1, math.ceil(len(mis) / PER_PAGE))
print(f"{len(mis)} misclassified images -> {n_pages} page(s) of up to {PER_PAGE}")

for page in range(n_pages):
    rows = mis.iloc[page * PER_PAGE:(page + 1) * PER_PAGE]
    if rows.empty:
        break

    fig, axes = plt.subplots(
        len(rows), 2, figsize=(17, 4.3 * len(rows)),
        gridspec_kw={"width_ratios": [1, 2.3]}, squeeze=False,
    )
    fig.suptitle(f"Misclassified examples - page {page + 1} of {n_pages}", fontsize=13)

    for (ax_img, ax_diag), (_, row) in zip(axes, rows.iterrows()):
        # ---- left: the strip image (same one the gallery shows) ----
        exact = row.get("Image_Path", "")
        if isinstance(exact, str) and exact and Path(exact).is_file():
            img_path = Path(exact)
        else:
            img_path = find_image(row["Image"], P["image_dirs"])
        img_bgr = cv2.imread(str(img_path)) if img_path is not None else None

        if img_bgr is None:
            ax_img.text(0.5, 0.5, f"Image not found\n{row['Image']}", ha="center", va="center")
        else:
            ax_img.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        ax_img.set_title(
            f"{row['Image']}  ({row.get('Image_Dir', '')})\n"
            f"Actual: {row['Actual_Class']}  |  Predicted: {row['Predicted_Class']}",
            fontsize=10, color="red",
        )
        ax_img.axis("off")

        # ---- right: the diagnostic plot saved in step 4 ----
        diag_path = row["Diagnostic_Plot"]
        if isinstance(diag_path, str) and Path(diag_path).is_file():
            ax_diag.imshow(mpimg.imread(diag_path))
        else:
            ax_diag.text(0.5, 0.5, "No diagnostic plot found", ha="center", va="center")
        ax_diag.axis("off")
        ax_diag.set_title(
            "Predicted concentration per band:  "
            + "   ".join(f"B{i}={row[f'Band_{i}_Prediction']:.2f}" for i in range(1, 7)),
            fontsize=10,
        )

    plt.tight_layout()
    out_png = cv_dir / f"misclassified_with_diagnostics_page{page + 1}.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

---
### Outputs

Everything is under `outputs/`:

| Path | Contents |
|------|----------|
| `calibration.json` | Band positions (mean/std, cm) |
| `datasets/band{1..6}.csv` | Training tables (`Control`, `Signal`, `Target`) |
| `models/final_model_band{1..6}.pth` | Trained regressors + scalers |
| `results/regressor_cv_metrics.csv`, `results/regressor_plots/` | Regressor CV results |
| `results/patient_results/patient_predictions.csv` | Per-image intensities, ratios, predictions |
| `results/patient_results/cv5/` | Classifier metrics, confusion matrix, misclassified images |

> Do not commit patient images, patient CSVs, or notebook outputs containing patient photos to a public repository.